In [ ]:
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    BitsAndBytesConfig,
)
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training, TaskType
from trl import SFTTrainer, SFTConfig
import json
import random
from tqdm import tqdm
import os

# Configuration
MODEL_ID = "Qwen/Qwen3.5-2B"
DATASET_PATH = "data/final_merged_dataset.jsonl"
OUTPUT_DIR = "outputs/qwen3.5-2b-lora"
MERGED_OUTPUT_DIR = "outputs/qwen3.5-2b-merged"
max_length = 2048

# Check GPU
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"BF16 supported: {torch.cuda.is_bf16_supported()}")

c:\Users\ALI\Desktop\finetune\qwen3_5\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch version: 2.11.0+cu130
CUDA available: True
GPU: NVIDIA GeForce RTX 5090
VRAM: 34.2 GB
BF16 supported: True


In [2]:
print(f"\nLoading model: {MODEL_ID}")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    padding_side="right",
)

# Ensure pad token is set
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

# Load model in 4-bit for QLoRA
compute_dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    # attn_implementation="flash_attention_2",  # Use Flash Attention 2 if available
)

model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

print(f"\nModel loaded successfully")
print(f"Compute dtype: {compute_dtype}")
print(f"Model device: {model.device}")


Loading model: Qwen/Qwen3.5-2B


W0422 11:06:31.941000 12636 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d
Loading weights:   0%|          | 1/320 [00:00<04:43,  1.13it/s]c:\Users\ALI\Desktop\finetune\qwen3_5\.venv\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100%|██████████| 320/320 [00:03<00:00, 84.67it/s] 



Model loaded successfully
Compute dtype: torch.bfloat16
Model device: cuda:0


In [3]:
# LoRA Configuration
# - r=64: Higher rank for diverse multi-domain dataset (code, math, cybersecurity, Turkish)
# - alpha=128: 2x rank for stable effective learning rate
# - dropout=0.05: Light regularization for large dataset
# - target_modules: Attention (q,k,v,o) + MLP (gate, up, down) for comprehensive adaptation

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    bias="none",
    inference_mode=False,
)

# Apply LoRA to model
model = get_peft_model(model, lora_config)

# Print trainable parameters
model.print_trainable_parameters()

trainable params: 21,823,488 || all params: 1,903,648,576 || trainable%: 1.1464


In [4]:
def to_chatml(system: str, user: str, assistant: str) -> str:
    """Convert to ChatML format for Qwen3.5"""
    text = f"<|im_start|>system\n{system}<|im_end|>\n"
    text += f"<|im_start|>user\n{user}<|im_end|>\n"
    text += f"<|im_start|>assistant\n{assistant}<|im_end|>"
    return text

def save_jsonl(data: list, filename: str):
    """Save list of dicts to JSONL file"""
    filepath = os.path.join("data", filename)
    with open(filepath, "w", encoding="utf-8") as f:
        for item in data:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")
    print(f"Saved {len(data):,} samples to {filepath}")
    return len(data)

MATH_SYSTEM_PROMPT = """You are an expert mathematician. Solve problems step-by-step with clear reasoning. Show your complete thought process before giving the final answer."""

In [5]:
ds = load_dataset(
      "nvidia/OpenMathReasoning",
      split="cot",
      cache_dir="C:/Users/ALI/.cache/huggingface/datasets"
  ).select(range(100000))
print(f"Loaded {len(ds):,} samples")


# Process Dataset
print("\nProcessing dataset to ChatML format...")

processed = []
skipped = 0

for row in tqdm(ds, desc="Processing OpenMathReasoning"):
    problem = row.get("problem", "")
    solution = row.get("generated_solution", "")

    # Skip empty entries
    if not problem or not solution:
        skipped += 1
        continue

    # Create ChatML formatted text
    text = to_chatml(
        system=MATH_SYSTEM_PROMPT,
        user=problem,
        assistant=solution
    )

    processed.append({"text": text})

print(f"\nProcessed: {len(processed):,} samples")
print(f"Skipped (empty): {skipped:,} samples")


# Save Full Dataset
count = save_jsonl(processed, "openmath_reasoning_full.jsonl")

# Qwen3.5 2B can train on full dataset, but subset is useful for quick experiments
save_jsonl(processed, "openmath_reasoning_100k.jsonl")

Loaded 100,000 samples

Processing dataset to ChatML format...


Processing OpenMathReasoning: 100%|██████████| 100000/100000 [00:07<00:00, 12692.76it/s]



Processed: 100,000 samples
Skipped (empty): 0 samples
Saved 100,000 samples to data\openmath_reasoning_full.jsonl
Saved 100,000 samples to data\openmath_reasoning_100k.jsonl


100000

In [6]:
import wandb
wandb.login()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\ALI\_netrc.
wandb: Currently logged in as: aliozcelik265 (aliozcelik265-radius-teknoloji) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [7]:
DATASET_PATH = "data/openmath_reasoning_100k.jsonl"

print(f"\nLoading dataset from: {DATASET_PATH}")
dataset = load_dataset("json", data_files=DATASET_PATH, split="train")
print(f"Dataset size: {len(dataset)} samples")


Loading dataset from: data/openmath_reasoning_100k.jsonl


Generating train split: 100000 examples [00:04, 22990.51 examples/s]


Dataset size: 100000 samples


In [8]:
# Training arguments for 32GB VRAM QLoRA
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,

    # Batch size settings
    per_device_train_batch_size=2,
    gradient_accumulation_steps=20,

    # Learning rate settings
    # - lr=2e-5: Conservative for large model stability
    # - warmup_ratio=0.03: 3% warmup for smooth start
    # - cosine scheduler: Gradual decay
    learning_rate=2e-5,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",

    # Training duration
    # - 1 epoch: Sufficient for large diverse dataset
    num_train_epochs=1,

    # Regularization
    weight_decay=0.01,

    # Precision
    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
    fp16=not (torch.cuda.is_available() and torch.cuda.is_bf16_supported()),

    # Logging
    logging_steps=1500,
    logging_first_step=True,

    # Saving
    save_strategy="steps",
    save_steps=500,
    save_total_limit=3,

    # Optimization
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    max_grad_norm=1.0,

    # Sequence length
    max_length=max_length,

    # Dataset
    dataset_text_field="text",
    packing=False,  # Disable packing for ChatML format

    # Misc
    report_to="wandb",  # Set to "wandb" if using Weights & Biases
    seed=42,

    # Disable neftune noise (optional, can enable for regularization)
    neftune_noise_alpha=None,
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [9]:
print("\nInitializing trainer...")

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
)

print("\nStarting training...")
print(f"  Total samples: {len(dataset)}")
print(f"  Batch size: {training_args.per_device_train_batch_size}")
print(f"  Gradient accumulation: {training_args.gradient_accumulation_steps}")
print(f"  Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"  Learning rate: {training_args.learning_rate}")
print(f"  Max sequence length: {max_length}")
print(f"  Epochs: {training_args.num_train_epochs}")


Initializing trainer...


Tokenizing train dataset: 100%|██████████| 100000/100000 [11:18<00:00, 147.47 examples/s]



Starting training...
  Total samples: 100000
  Batch size: 2
  Gradient accumulation: 20
  Effective batch size: 40
  Learning rate: 2e-05
  Max sequence length: 2048
  Epochs: 1


In [10]:
# Trains
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046, 'pad_token_id': 248044}.


Step,Training Loss
1,0.670834
1500,0.537409


TrainOutput(global_step=2500, training_loss=0.5323709702014923, metrics={'train_runtime': 111817.4828, 'train_samples_per_second': 0.894, 'train_steps_per_second': 0.022, 'total_flos': 1.7133065773304387e+18, 'train_loss': 0.5323709702014923})

In [11]:
print("\nTraining complete!")

# %% Cell 7: Save Model

print(f"\nSaving LoRA adapter to: {OUTPUT_DIR}")

# Save the LoRA adapter
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("LoRA adapter saved successfully!")

# Print saved files
print("\nSaved files:")
for f in os.listdir(OUTPUT_DIR):
    filepath = os.path.join(OUTPUT_DIR, f)
    size = os.path.getsize(filepath) / 1e6
    print(f"  {f}: {size:.1f} MB")


Training complete!

Saving LoRA adapter to: outputs/qwen3.5-2b-lora
LoRA adapter saved successfully!

Saved files:
  adapter_config.json: 0.0 MB
  adapter_model.safetensors: 43.7 MB
  chat_template.jinja: 0.0 MB
  checkpoint-1500: 0.0 MB
  checkpoint-2000: 0.0 MB
  checkpoint-2500: 0.0 MB
  README.md: 0.0 MB
  tokenizer.json: 20.0 MB
  tokenizer_config.json: 0.0 MB


In [17]:
from peft import PeftModel


print(f"\nMerging LoRA adapter into base model: {OUTPUT_DIR}")

MERGED_OUTPUT_DIR = "outputs/qwen3.5-2b-merged"

merge_dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16

# Free the 4-bit training model before loading a mergeable base model.
#del trainer
#del model
torch.cuda.empty_cache()

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=compute_dtype,
    device_map="auto",
    trust_remote_code=True,
)

merged_model = PeftModel.from_pretrained(base_model, OUTPUT_DIR)
merged_model = merged_model.merge_and_unload()

merged_model.save_pretrained(MERGED_OUTPUT_DIR)
tokenizer.save_pretrained(MERGED_OUTPUT_DIR)

print("Merged model saved successfully!")
print(f"Saved merged model to: {MERGED_OUTPUT_DIR}")



Merging LoRA adapter into base model: outputs/qwen3.5-2b-lora


Writing model shards: 100%|██████████| 1/1 [00:03<00:00,  3.23s/it]

Merged model saved successfully!
Saved merged model to: outputs/qwen3.5-2b-merged


In [18]:
print("\nMerged model files:")
for f in os.listdir(MERGED_OUTPUT_DIR):
    filepath = os.path.join(MERGED_OUTPUT_DIR, f)
    size = os.path.getsize(filepath) / 1e6
    print(f"  {f}: {size:.1f} MB")



Merged model files:
  chat_template.jinja: 0.0 MB
  config.json: 0.0 MB
  generation_config.json: 0.0 MB
  model.safetensors: 3763.7 MB
  tokenizer.json: 20.0 MB
  tokenizer_config.json: 0.0 MB
